# WIA3003: Academic Project II
## Cross-Validated Performance by Recorded Ethnicity

Notebook 11 scored each ethnicity group on one test split of 210 records. Notebook 13 showed those subgroup differences were mostly noise for logistic regression. This notebook repeats the subgroup evaluation with five-fold grouped cross-validation, so every record is tested once and each group has 282 to 380 test records.

Models: the initial QCNN from notebook 04 and logistic regression from notebook 07. The QCNN keeps its saved configuration, including the 13 epochs selected in notebook 04. No settings are tuned here. The Streamlit app reads the saved table. Claude assisted with writing and running this notebook.

## 1.0 Load Dataset and Folds

In [1]:
import hashlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pennylane as qml
import sklearn
from pennylane import numpy as pnp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler

folder = Path.cwd()
project_folder = folder.parent if folder.name == "notebooks" else folder
details = json.loads((project_folder / "results" / "qcnn_baseline" / "run_details.json").read_text(encoding="utf-8"))

data_path = Path.home() / "Downloads" / "Orbital, Nasal and Craniofacial Class Pattern.xls"
dataset_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()
if dataset_hash != details["dataset_sha256"]:
    raise ValueError("Workbook changed since the grouped QCNN run. Verify the data first.")

df = pd.read_excel(data_path, sheet_name="Raw Data")
df.columns = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()
features = details["features"]
X = df[features].apply(pd.to_numeric, errors="raise")
y = df["Gender"].astype("string").str.strip().map(details["label_mapping"]).astype(int).to_numpy()
ethnicity = df["Ethnicity"].astype("string").str.strip().str.upper().astype(str).to_numpy()
groups = X.groupby(features, sort=False, dropna=False).ngroup().to_numpy()

FOLD_SEED = 42
strata = np.char.add(y.astype(str), ethnicity)
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=FOLD_SEED)
fold_rows = list(splitter.split(X, strata, groups))
fold_of = np.empty(len(X), dtype=int)
for fold, (train_rows, test_rows) in enumerate(fold_rows):
    assert not set(groups[train_rows]) & set(groups[test_rows])
    fold_of[test_rows] = fold
print("Records per fold:", np.bincount(fold_of).tolist())

Records per fold: [209, 209, 208, 208, 208]


## 2.0 Initial QCNN

Same circuit and training settings as notebook 04: eight qubits, two shared convolution layers, CRY pooling, fixed-average readout, Adam with learning rate 0.02, batch size 32 and seed 42. Each fold fits its own scaler on the training records only.

In [2]:
SEED = details["seed"]
EPOCHS = details["selected_epochs"]
BATCH_SIZE = details["batch_size"]
LEARNING_RATE = details["learning_rate"]
n_qubits = 8

def convolution_layer(parameters):
    for first in [0, 2, 4, 6, 1, 3, 5]:
        qml.Rot(*parameters[:3], wires=first)
        qml.Rot(*parameters[3:], wires=first + 1)
        qml.CNOT(wires=[first, first + 1])

@qml.qnode(qml.device("default.qubit", wires=n_qubits), interface="autograd", diff_method="backprop")
def qcnn_circuit(inputs, weights):
    for wire in range(n_qubits):
        qml.RY(inputs[..., wire], wires=wire)
    convolution_layer(weights[:6])
    convolution_layer(weights[6:12])
    for source in [0, 2, 4, 6]:
        qml.CRY(weights[12], wires=[source, source + 1])
    return [qml.expval(qml.PauliZ(wire)) for wire in [1, 3, 5, 7]]

def predict_probability(parameters, inputs):
    return (1 - pnp.mean(pnp.stack(qcnn_circuit(inputs, parameters)), axis=0)) / 2

def loss_function(parameters, inputs, labels):
    probabilities = pnp.clip(predict_probability(parameters, inputs), 1e-7, 1 - 1e-7)
    return -pnp.mean(labels * pnp.log(probabilities) + (1 - labels) * pnp.log(1 - probabilities))

def fit_predict_qcnn(train_rows, test_rows):
    scaler = MinMaxScaler(feature_range=(0, np.pi), clip=True)
    inputs = pnp.array(scaler.fit_transform(X.iloc[train_rows]), requires_grad=False)
    labels = pnp.array(y[train_rows], requires_grad=False)
    weights = pnp.array(np.random.default_rng(SEED).uniform(-0.1, 0.1, 13), requires_grad=True)
    optimiser = qml.AdamOptimizer(stepsize=LEARNING_RATE)
    rng = np.random.default_rng(SEED)
    for _ in range(EPOCHS):
        order = rng.permutation(len(inputs))
        for start in range(0, len(order), BATCH_SIZE):
            batch = order[start:start + BATCH_SIZE]
            weights = optimiser.step(lambda current: loss_function(current, inputs[batch], labels[batch]), weights)
    test_inputs = pnp.array(scaler.transform(X.iloc[test_rows]), requires_grad=False)
    return np.asarray(predict_probability(weights, test_inputs), dtype=float)

def fit_predict_logistic(train_rows, test_rows):
    model = Pipeline([("scaler", StandardScaler()),
                      ("logistic", LogisticRegression(solver="lbfgs", C=1.0, max_iter=1000))])
    model.fit(X.iloc[train_rows], y[train_rows])
    return model.predict_proba(X.iloc[test_rows])[:, 1]

## 3.0 Run Cross-Validation

In [3]:
models = {"Initial QCNN": fit_predict_qcnn, "Logistic regression": fit_predict_logistic}
probability = {name: np.empty(len(X)) for name in models}
for fold, (train_rows, test_rows) in enumerate(fold_rows):
    for name, fit_predict in models.items():
        started = time.perf_counter()
        probability[name][test_rows] = fit_predict(train_rows, test_rows)
        print(f"Fold {fold + 1}: {name} done in {time.perf_counter() - started:.0f} s", flush=True)
predicted = {name: (values >= 0.5).astype(int) for name, values in probability.items()}

Fold 1: Initial QCNN done in 78 s


Fold 1: Logistic regression done in 0 s


Fold 2: Initial QCNN done in 66 s


Fold 2: Logistic regression done in 0 s


Fold 3: Initial QCNN done in 69 s


Fold 3: Logistic regression done in 0 s


Fold 4: Initial QCNN done in 67 s


Fold 4: Logistic regression done in 0 s


Fold 5: Initial QCNN done in 73 s


Fold 5: Logistic regression done in 0 s


## 4.0 Results by Recorded Ethnicity

Every record appears once as a test record. Columns match notebook 11 so the two tables can be compared.

In [4]:
def score(model, group, rows):
    labels, guesses = y[rows], predicted[model][rows]
    correct, female, male = labels == guesses, labels == 0, labels == 1
    return {
        "Model": model, "Recorded ethnicity": group, "Test records": int(rows.sum()),
        "Correct": int(correct.sum()), "Accuracy": correct.mean(),
        "Female records": int(female.sum()), "Female correct": int((correct & female).sum()),
        "Female recall": correct[female].mean(),
        "Male records": int(male.sum()), "Male correct": int((correct & male).sum()),
        "Male recall": correct[male].mean(),
        "Macro F1": f1_score(labels, guesses, average="macro"),
    }

results = pd.DataFrame([score(model, group, ethnicity == group)
                        for model in models for group in sorted(set(ethnicity))])
overall = pd.DataFrame([score(model, "ALL", np.ones(len(X), dtype=bool)) for model in models])
display(results.round(4))
display(overall.round(4))

fold_accuracy = pd.DataFrame({
    model: [np.mean(predicted[model][fold_of == fold] == y[fold_of == fold]) for fold in range(5)]
    for model in models})
print("Accuracy across folds (mean, std):")
display(fold_accuracy.agg(["mean", "std"]).round(4))

,Model,Recorded ethnicity,Test records,Correct,Accuracy,Female records,Female correct,Female recall,Male records,Male correct,Male recall,Macro F1
0,Initial QCNN,CHINESE,380,266,0.7000,176,101,0.5739,204,165,0.8088,0.6912
1,Initial QCNN,INDIAN,282,205,0.7270,100,70,0.7000,182,135,0.7418,0.7116
2,Initial QCNN,MALAY,380,247,0.6500,179,87,0.4860,201,160,0.7960,0.6366
3,Logistic regression,CHINESE,380,292,0.7684,176,125,0.7102,204,167,0.8186,0.7656
4,Logistic regression,INDIAN,282,224,0.7943,100,76,0.7600,182,148,0.8132,0.7800
5,Logistic regression,MALAY,380,286,0.7526,179,128,0.7151,201,158,0.7861,0.7511


,Model,Recorded ethnicity,Test records,Correct,Accuracy,Female records,Female correct,Female recall,Male records,Male correct,Male recall,Macro F1
0,Initial QCNN,ALL,1042,718,0.6891,455,258,0.5670,587,460,0.7836,0.6769
1,Logistic regression,ALL,1042,802,0.7697,455,329,0.7231,587,473,0.8058,0.7652


Accuracy across folds (mean, std):


,Initial QCNN,Logistic regression
mean,0.6890,0.7697
std,0.0368,0.0324


## 5.0 Save Results

In [5]:
output_folder = project_folder / "results" / "ethnicity_evaluation"
output_folder.mkdir(parents=True, exist_ok=True)
results.to_csv(output_folder / "cross_validated_subgroup_metrics.csv", index=False)
overall.to_csv(output_folder / "cross_validated_overall_metrics.csv", index=False)
pd.DataFrame({"source_row_position": np.arange(len(X)), "fold": fold_of, "actual_label": y,
              **{f"{name} probability_male": values for name, values in probability.items()}}
             ).to_csv(output_folder / "cross_validated_predictions.csv", index=False)
settings = {
    "dataset_sha256": dataset_hash, "features": features, "folds": 5, "fold_seed": FOLD_SEED,
    "method": "StratifiedGroupKFold, strata sex x ethnicity, groups of identical measurements",
    "qcnn": {"epochs": EPOCHS, "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE, "seed": SEED,
             "readout": "fixed average"},
    "threshold": 0.5, "pennylane_version": qml.__version__, "sklearn_version": sklearn.__version__,
}
(output_folder / "cross_validation_settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
print("Saved results to:", output_folder)

Saved results to: C:\Users\jwenn\Desktop\UM\FYP\FYP-Project\results\ethnicity_evaluation
